<a href="https://colab.research.google.com/github/harshangs/Hackathon/blob/main/starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WarehouseSort — Starter Notebook

This notebook walks through the **image IL pipeline** end-to-end on the easy level:
install → look at the env → download demos → train an RGB Diffusion Policy → evaluate.

This is an **image challenge**: the policy sees only a scene-camera RGB image + robot
proprioception. The provided RGB Diffusion Policy is a **runnable template** — it does **not**
yet solve the task; making an image policy work is the point.

**Requirements:** a CUDA GPU (ManiSkill 3 runs on GPU). In Google Colab, go to
*Runtime → Change runtime type → T4 GPU*.

References:
- [ManiSkill 3](https://maniskill.readthedocs.io/en/latest/) — GPU-accelerated robot simulation
- [Diffusion Policy](https://diffusion-policy.cs.columbia.edu) — Chi et al. 2023

### What a solved episode looks like

The scripted policy (used only to generate the demos) sorting parcels into the
color-matched bins — left panel is the scene view, right is the policy's camera:

![easy demo](https://github.com/marso-robotics/berlin-marso-hackathon/raw/main/media/easy_demo.gif)

*(medium = 4 parcels, hard = 6 parcels with bins that may swap sides — see the README.)*

## 1. Install

In [ ]:
# Install ManiSkill and dependencies (takes ~2 min on Colab)
!pip install mani-skill==3.0.1 diffusers==0.38.0 gymnasium torch torchvision hydra-core -q

# Clone the repo (skip if already in the repo directory)
import os
if not os.path.exists('warehouse_sort'):
    !git clone https://github.com/marso-robotics/berlin-marso-hackathon.git
    %cd berlin-marso-hackathon

# Install the package
!pip install -e . -q

# Colab headless rendering (offscreen Vulkan)
import os
os.environ['DISPLAY'] = ''
os.environ['PYOPENGL_PLATFORM'] = 'egl'

## 2. Look at the environment

**Easy level**: 2 parcels (1 red-tagged, 1 blue-tagged), 2 color-coded bins, fixed positions.
The observation is the **scene-camera RGB image** + robot **proprioception** only — no parcel
or bin coordinates. Your policy must read the colors from pixels.

In [ ]:
import gymnasium as gym
import torch
import warehouse_sort  # registers WarehouseSort-v1
from mani_skill.utils.wrappers.flatten import FlattenRGBDObservationWrapper

env = gym.make(
    'WarehouseSort-v1', num_envs=1, obs_mode='rgb',
    control_mode='pd_ee_delta_pos', sim_backend='gpu', render_mode='rgb_array',
    difficulty='easy', num_parcels=2, fixed_poses=True,
)
env = FlattenRGBDObservationWrapper(env, rgb=True, depth=False, state=True)

obs, _ = env.reset(seed=42)
print('obs keys:', list(obs.keys()))
print('  rgb  :', tuple(obs['rgb'].shape), obs['rgb'].dtype)     # (1,128,128,3) uint8
print('  state:', tuple(obs['state'].shape), obs['state'].dtype) # (1,26) proprioception only
print('action space:', env.action_space)                 # Box(-1,1,(4,))
print()
print('The policy sees ONLY the image + proprioception — no parcel/bin coordinates.')

In [ ]:
# Render the scene
from IPython.display import Image as IPImage
import PIL.Image, io

frame = env.render()                    # (1, H, W, 3) uint8
img = PIL.Image.fromarray(frame[0].cpu().numpy())
buf = io.BytesIO()
img.save(buf, format='PNG')
IPImage(buf.getvalue())

In [ ]:
# Run the scripted waypoint policy to verify the env works (it generates the demos;
# it reads privileged sim state to control the arm, so it is NOT a submittable policy).
import sys
sys.path.insert(0, '.')
from examples.scripted_policy import scripted_episode

env.close()
env = gym.make('WarehouseSort-v1', num_envs=1, obs_mode='rgb',
               control_mode='pd_ee_delta_pos', sim_backend='gpu',
               difficulty='easy', num_parcels=2, fixed_poses=True, max_episode_steps=200)
history = scripted_episode(env, max_steps=200, seed=42)
final_info = history[-1][-1]
print(f"Sorted: {final_info['success_count'].item():.0f} / 2")
env.close()

## 3. Demonstrations (provided — Kaggle competition data)

The rgb demos (**200 episodes per level**) are the competition data:
[the data tab](https://www.kaggle.com/competitions/marso-hack-berlin-2026-robot-parcel-sorting-challenge/data). You don't need to record any.

- **On Kaggle** (notebook attached to the competition): the data is already mounted under
  `/kaggle/input/` — nothing to set up; the cell below finds it.
- **On Colab / local**: the cell downloads it with `kagglehub`, which needs your Kaggle API
  credentials (one-time). First **join the competition** (Rules → *I Understand and Accept*),
  then get a token: on kaggle.com click your **avatar → Settings → API → Create New API Token**.
  (Ensure you use the legacy method to download `kaggle.json` containing your `username` and `key`). Then set them in the cell —
  the auth lines are at the top, just uncomment and paste your values.

Either way the cell stages the files under `il/demos/<level>/` so the training commands below
work unchanged.

> ⚠️ The demos come from a scripted policy. Using it to collect *data* is fine, but submitting a
> scripted / hard-coded controller (or any policy that reads privileged env state) is
> **disqualified** — your submitted policy must act from the observation.

In [ ]:
# The rgb demos (200 episodes per level) are the Kaggle COMPETITION data.
COMPETITION = 'marso-hack-berlin-2026-robot-parcel-sorting-challenge'

import os, glob, shutil, tarfile

# --- Colab / local auth (NOT needed on Kaggle) -------------------------------------------
# After joining the competition, get a token from kaggle.com -> Settings -> API ->
# 'Create New API Token' (downloads kaggle.json), then uncomment and paste its values:
# os.environ['KAGGLE_USERNAME'] = 'your_kaggle_username'
# os.environ['KAGGLE_KEY']      = 'your_kaggle_key'      # the 'key' field in kaggle.json
# (Alternatively run  `import kagglehub; kagglehub.login()`  for an interactive prompt.)
# -----------------------------------------------------------------------------------------

# 1) locate the data: a Kaggle-mounted input, else download via kagglehub
src = next((p for p in glob.glob('/kaggle/input/*')
           if glob.glob(os.path.join(p, '**/trajectory.rgb.*.h5'), recursive=True)
           or glob.glob(os.path.join(p, '**/*.tar.gz'), recursive=True)), None)
if src is None:
    import kagglehub
    src = kagglehub.competition_download(COMPETITION)
print('data at:', src)

# 2) stage into il/demos/<level>/  (handles a tarball OR an easy/medium/hard folder layout)
os.makedirs('il/demos', exist_ok=True)
tars = glob.glob(os.path.join(src, '**/*.tar.gz'), recursive=True)
if tars:
    for t in tars:
        with tarfile.open(t) as tf: tf.extractall('il/demos')
else:
    for h5 in glob.glob(os.path.join(src, '**/trajectory.rgb.*.h5'), recursive=True):
        lvl = os.path.basename(os.path.dirname(h5))
        os.makedirs(f'il/demos/{lvl}', exist_ok=True)
        for f in glob.glob(h5[:-2] + '*'):   # the .h5 and its .json
            shutil.copy(f, f'il/demos/{lvl}/{os.path.basename(f)}')
print('staged levels:', sorted(os.path.basename(os.path.dirname(x))
                               for x in glob.glob('il/demos/*/trajectory.rgb.*.h5')))

In [ ]:
# Check the rgb demos are present
import glob
for f in sorted(glob.glob('il/demos/easy/*.rgb*.h5')):
    print(f)

### 📈 Watch training live with TensorBoard

Run the two lines below **before** the training cell. `%tensorboard` starts a background
server and embeds a dashboard that **auto-refreshes as training writes metrics** (loss,
eval sort accuracy), so you can keep watching while the training cell runs.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir il/baselines/diffusion_policy/runs

## 4. Train the RGB Diffusion Policy

[Diffusion Policy](https://diffusion-policy.cs.columbia.edu) (Chi et al. 2023) with an image
front-end (ResNet18 + SpatialSoftmax) → ConditionalUNet1D. A plain MLP behavior cloner fails
due to compounding error; DP's action chunking helps — but note the image policy is a
**template that does not yet solve the task**.

This uses a short run (`total_iters=10000`) to verify the pipeline. **For real training scale
up to `total_iters=30000` or more** (~30–90 min on a T4).

> The rgb observation has the same shape at every difficulty, so **one trained checkpoint can
> run on easy, medium, and hard**.

In [ ]:
# Quick training run (verify pipeline; not fully converged)
!python il/train.py method=dp_rgb demo_dir=easy \
    flags.total_iters=10000 \
    flags.eval_freq=5000 \
    flags.exp_name=warehouse_rgb_dp_starter

In [ ]:
# For real training (uncomment and run)
# !python il/train.py method=dp_rgb demo_dir=easy \
#     flags.total_iters=30000 \
#     flags.eval_freq=5000 \
#     flags.exp_name=warehouse_rgb_dp

## 5. Evaluate the checkpoint

In [ ]:
import glob, os

# Find the latest checkpoint
ckpts = sorted(glob.glob(
    'il/baselines/diffusion_policy/runs/warehouse_rgb_dp_starter/checkpoints/*.pt'
))
if not ckpts:
    print('No checkpoint found — run training first')
else:
    ckpt = ckpts[-1]
    print(f'Using checkpoint: {ckpt}')
    !python eval.py difficulty=easy \
        policy=warehouse_sort.il_policy:load_dp_rgb \
        checkpoint={ckpt} \
        eval_config=conf/eval/default.yaml

**Watch the rollout.** `eval.py` prints the metrics above and saves a rollout video (render + policy-camera views). Display it below — with the untrained template the arm mostly flails, which is exactly the gap you're closing.

In [ ]:
# Display the eval rollout video (eval.py saves one under outputs/<date>/<time>/videos/)
import glob, os
from IPython.display import Video, display
vids = sorted(glob.glob('outputs/**/videos/*.mp4', recursive=True), key=os.path.getmtime)
if vids:
    print('eval rollout:', vids[-1])
    display(Video(vids[-1], embed=True, width=640))
else:
    print('No eval video found — run the eval cell above first.')

## 6. Scaling to medium and hard

The rgb observation has the **same shape at every difficulty**, so you can evaluate the same
checkpoint on medium and hard with no retraining — or train per level. The env, policy
entrypoint, and eval command are identical; only `difficulty=` changes.

```bash
# evaluate the SAME checkpoint on medium / hard
python eval.py difficulty=medium policy=warehouse_sort.il_policy:load_dp_rgb \
    checkpoint=<ckpt> eval_config=conf/eval/default.yaml
python eval.py difficulty=hard   policy=warehouse_sort.il_policy:load_dp_rgb \
    checkpoint=<ckpt> eval_config=conf/eval/default.yaml

# or train per level
python il/train.py method=dp_rgb demo_dir=medium flags.exp_name=warehouse_rgb_dp_medium
python il/train.py method=dp_rgb demo_dir=hard   flags.exp_name=warehouse_rgb_dp_hard
```

## 8. Next steps — how to submit

You've trained and evaluated a policy. From here:

1. **Push to medium and hard** (§6) — they carry the most weight (medium 0.3, hard 0.5).
2. **Or bring your own approach** — any policy that implements the
   `act(obs, deterministic=True)` contract works (RL, scripted, transformer, ...).
3. **Package your entry** — you submit a **GitHub repo** containing your policy
   entrypoint, your checkpoint(s), and a `submission.yaml` declaring the levels +
   checkpoint for each.

**Read [SUBMISSION.md](https://github.com/marso-robotics/berlin-marso-hackathon/blob/main/SUBMISSION.md)**
for the full guide: how the code ties together, the policy contract (with a minimal
example), how to declare what gets scored, and exactly how to submit.

> If you're on Colab, `SUBMISSION.md` and `submission.example.yaml` are also in the
> repo you cloned in step 1 — open them from the file browser on the left.


**What you submit (3 things):** your **codebase** (this repo, forked) · a **`submission.yaml`** (per-level checkpoint) · your **checkpoint(s)** plus the **policy entrypoint** `module:function` that loads them into `act(obs)`.

### Ideas to improve the baseline

- **Train longer / more data**: raise `flags.total_iters`; record more demos with `il/gen_demos.py`.
- **Horizons**: `flags.pred_horizon`, `flags.act_horizon`, `flags.obs_horizon`.
- **Eval denoising steps**: `num_inference_steps` in `load_dp_rgb` (more = better, slower).
- **Capacity**: `unet_dims`, `diffusion_step_embed_dim`; **optimisation**: `flags.batch_size`, LR.
- **Generalise** to the held-out wider positions / bin-swaps (hard is weighted 0.5).

> ⚠️ If you change an architecture/horizon hyperparameter for training, pass the **same** value to your policy loader (`load_dp_rgb(...)`) or the checkpoint won't load.